# Un core · M original · A, tau_r y tau_d por separado

Ya existe una sensibilidad sólo de A en el10 (sin O₂, liberación2.348h⁻¹). El09 E2 ajustó conjuntamente los tres parámetros y otros coeficientes: no separaba sus efectos. Este ensayo completa la sensibilidad individual con la referencia de transferencia del11: **O₂ activo, liberación0.5870419885h⁻¹, escalaCsat2.5, ganancia0.77428737694**, theta_nv2 y todos los demás parámetros fijos. No se utiliza liberación transitoria.

Se conserva el M original, incluida su dependencia lineal con dosis:
\[
M(t)=1+A\sum_p\frac{\Delta N_p}{0.14}
\frac{(1-e^{-u_p/\tau_r})e^{-u_p/\tau_d}}{h_{\max}},\quad u_p=\max(t-t_p,0).
\]
El máximo analítico del kernel aislado ocurre en \(u_*=\tau_r\ln(1+\tau_d/\tau_r)\). La normalización mantiene su máximo en1 al variar los tiempos: cambiar tau modifica cronología y área del refuerzo, no sólo su posición.

Referencia A=0.3,tau_r=4h,tau_d=12h. Familias independientes:
- Sólo A:0,0.1,0.2, manteniendo tiempos4/12h.
- Sólo tau_r:1,2,8h, manteniendo A=0.3,tau_d=12h.
- Sólo tau_d:4,8,24h, manteniendo A=0.3,tau_r=4h.

A=0 elimina M; por eso las pruebas de tiempos se realizan con A=0.3 y no con A=0. No se selecciona ni optimiza un escenario con ningún LAB. LAB019–021 y LAB016–018 se rotulan por su papel previo de calibración/holdout, no como nueva validación independiente.

Una sola química nutrida por LAB, mismos eventos/iniciales/temperaturas/horizontes de11 y Cd0=0. O₂ usa sólo X de esa química; no existe core0. Se comprueban parámetros constantes, balance y referencia A=0 frente a11. Cada alternativa tiene sus propias figuras de registro completo, pico24h tras pulso y cola desde32h; sólo referencia, alternativa y mediciones por panel, con leyendas superiores. Medianas horarias de RAW convertida por volumen remanente, sin filtrar la predicción. Se conserva LAB017 como un pulso0.28g/L.

La fuente es gate·actividad·fO₂·M·(44.01/46.07)(betaG+betaF)X. Depósito y liberación históricos constantes, sin el ensayo descartado12. Todo permanece dentro de SOURCE/EXECUTED; no se exportan CSV/PNG ni se modifican modelos, datos, parámetros o notebooks anteriores.


In [ ]:
from pathlib import Path
import sys,json,hashlib
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from dataclasses import replace
from IPython.display import display,Markdown
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/wine_fermentation').is_dir())
assert Path(sys.prefix).name.lower()=='fermentation'
sys.path.insert(0,str(ROOT/'src'));sys.dont_write_bytecode=True
from wine_fermentation.models.historical import co2 as gas,kinetics
from wine_fermentation.models.historical.calibration import load,release_forward,Search
from wine_fermentation.comparison import observations
NAME='13_single_core_original_M_sensitivity'
protected=[p for folder in ('src','configs','data','notebooks','results') for p in (ROOT/folder).rglob('*') if p.is_file() and '__pycache__' not in str(p) and not p.name.startswith(NAME+'.')]
before={p:hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
old=__import__('nbformat').read(ROOT/'notebooks/diagnostics/09_historical_single_core.executed.ipynb',4)
stream=''.join(o.get('text','') for o in old.cells[4].outputs if o.output_type=='stream')
records=json.JSONDecoder().raw_decode(stream.split('PARAMETROS_COMPLETOS_SIN_EXPORTACION\n')[1])[0]
p0=next(r['parametros'] for r in records if r['caso']=='Propuesta · un core, sin O₂' and r['etapa']=='E1')
assert p0['A']==.3
theta=kinetics.normalize_parameters(json.loads((ROOT/'configs/theta/theta_nv2.json').read_text()))
theta_before=theta.copy();assert 0<theta['Yeg']<=.5114 and 0<theta['Yef']<=.5114
v=load(ROOT);LABS=['LAB019','LAB020','LAB021','LAB016','LAB017','LAB018']
batches=dict(v.batches);b=batches['LAB017'];events=b.pulses['N'];assert np.isclose(sum(d for t,d in events),.28)
batches['LAB017']=replace(b,pulses={**b.pulses,'N':((min(t for t,d in events),.28),)})
raw=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',usecols=['lab','time_h','record_kind','co2_raw_g_l_h'])
obs=observations(raw);support={lab:obs[lab].loc[obs[lab].time_h.between(0,v.horizons[lab])].copy() for lab in LABS}
contexts={};calls=0
for lab,b in batches.items():
 f=gas.integrate_core(b,theta,b.time);calls+=1
 consumed=(f.G+f.F).iloc[0]-(f.G+f.F).to_numpy();z=np.clip((consumed-2.)/8.,0,1)
 contexts[lab]=dict(time=f.index.to_numpy(),dt=np.diff(f.index),gate=z*z*(3-2*z),q=gas.production(theta,b,f),
  sat_base=gas.csat(b,f,{'CO2sat_scale':1.}),events=b.pulses['N'],core=f)
assert calls==6 and all('q0' not in x and 'core0' not in x for x in contexts.values())
display(pd.Series(theta,name='theta_nv2 congelado'))
display(pd.Series(p0,name='Referencia E1; ahora A se fija en cero'))
windows={}
for lab in LABS:
 pulse=batches[lab].pulses['N'][0][0];end=v.horizons[lab]
 windows[lab]={'Registro':(0,end),'Arranque':(0,pulse),'Pico posterior':(pulse,min(pulse+24,end)),'Cola tardía':(pulse+32,end)}
display(pd.DataFrame([dict(LAB=lab,eventos=batches[lab].pulses['N'],corte=v.horizons[lab],**batches[lab].initials) for lab in LABS]))
display(pd.DataFrame([dict(LAB=lab,tramo=w,inicio=ab[0],fin=ab[1]) for lab,ww in windows.items() for w,ab in ww.items()]))


In [ ]:
base={**p0,'A':0.}
levels=[base['kCO2_release_h'],base['kCO2_release_h']/2,base['kCO2_release_h']/4]
oxygen={};factors={};o2audit=[]
for lab,c in contexts.items():
 b=batches[lab];t=c['time'];f=c['core'];O=np.zeros(len(t));ini=b.initials
 O[0]=base['O2_initial_scale']*gas.o2_saturation_mg_l(b.temperature_c[0],ini['E'],ini['G'],ini['F'])
 for i,dt in enumerate(c['dt']):
  uptake=base['O2_qmax_mg_gdw_h']*f.X.iloc[i]*O[i]/(.25+O[i])
  O[i+1]=max(O[i]-dt*uptake,0.)
 factor=.08+.92*.75**2/(.75**2+O**2)
 oxygen[lab]=O;factors[lab]=factor
 hit=np.flatnonzero(factor>=.99)
 o2audit.append(dict(LAB=lab,O2_inicial_mg_L=O[0],factor_inicial=factor[0],
  primera_hora_factor_099=float(t[hit[0]]) if len(hit) else np.nan,
  factor_al_pulso=float(np.interp(c['events'][0][0],t,factor))))
display(pd.DataFrame(o2audit))

base={**base,'A':.3,'kCO2_release_h':base['kCO2_release_h']/4}
REF='Referencia: A=.3, tau_r=4, tau_d=12'
scenarios={REF:base.copy()};families={}
for key,vals in [('A',[0.,.1,.2]),('tau_rise_h',[1.,2.,8.]),('tau_decay_h',[4.,8.,24.])]:
 for val in vals:
  label=f'{key}={val:g}';scenarios[label]={**base,key:val};families[label]=key
for label,p in scenarios.items():
 changed=[k for k in base if base[k]!=p[k]]
 assert changed==([] if label==REF else [families[label]])
display(pd.Series(base,name='Referencia fija; sólo se altera el parámetro de cada escenario'))
display(pd.DataFrame([dict(escenario=label,A=p['A'],tau_r=p['tau_rise_h'],tau_d=p['tau_decay_h'],
 pico_kernel_h=p['tau_rise_h']*np.log1p(p['tau_decay_h']/p['tau_rise_h'])) for label,p in scenarios.items()]))
curves={};rows=[];errors=[]
for label,p in scenarios.items():
 for lab,c in contexts.items():
  t=c['time'];r,M=Search.histories(t,c['events'],p);activity=1+(p['pulse_activity_gain']-1)*r
  src=c['gate']*activity*factors[lab]*M*c['q']
  pred,cd,qgas=release_forward({**c,'source':src},p)
  err=np.max(abs(np.diff(cd)-c['dt']*(src[:-1]-qgas[1:])))
  assert err<1e-9 and cd.min()>=0 and qgas.min()>-1e-12
  errors.append(err)
  curves[label,lab]=pd.DataFrame(dict(qpred=pred,Cd=cd,qprod=src,M=M),index=t)
  o=support[lab]
  for window in ['Registro','Pico posterior','Cola tardía']:
   start,end=windows[lab][window];z=o.loc[o.time_h.between(start,end)]
   tt=z.time_h.to_numpy();y=z.co2_raw_g_l_h.to_numpy();yp=np.interp(tt,t,pred);e=yp-y
   ce=np.where(y<.05,np.maximum(yp-.05,0),e)
   rows.append(dict(escenario=label,LAB=lab,rol='Calibración previa' if lab in LABS[:3] else 'Holdout previo',
    tramo=window,n=len(tt),RMSE=np.sqrt(np.mean(e*e)),RMSE_censurado=np.sqrt(np.mean(ce*ce)),sesgo=e.mean(),
    pico_predicho=yp.max(),pico_medido=y.max(),hora_pico_predicho=tt[np.argmax(yp)],hora_pico_medida=tt[np.argmax(y)],
    area_predicha=np.trapz(yp,tt),area_medida=np.trapz(y,tt)))
metrics=pd.DataFrame(rows)
with pd.option_context('display.max_rows',200):display(metrics.round(6))
summary=metrics.groupby(['tramo','rol','escenario'])[['RMSE','sesgo','pico_predicho']].mean()
display(summary.round(6))
ref0=metrics.loc[metrics.escenario.eq('A=0')&metrics.tramo.eq('Registro')].groupby('rol').RMSE.mean()
assert np.isclose(ref0['Calibración previa'],.1306462185,atol=1e-8)
assert np.isclose(ref0['Holdout previo'],.1082467127,atol=1e-8)


In [ ]:
for label in families:
 for window in ['Registro','Pico posterior','Cola tardía']:
  fig,axes=plt.subplots(2,3,figsize=(16,9))
  for ax,lab in zip(axes.flat,LABS):
   start,end=windows[lab][window];o=support[lab];o=o.loc[o.time_h.between(start,end)]
   ax.plot(o.time_h,o.co2_raw_g_l_h,color='black',lw=1.1,label='Medianas horarias del sensor')
   for name,color,ll in [(REF,'#0072B2','Referencia A=.3, tau_r=4, tau_d=12'),(label,'#009E73',label)]:
    f=curves[name,lab];m=(f.index>=start)&(f.index<=end)
    ax.plot(f.index[m],f.qpred[m],color=color,lw=1.6,label=ll)
   for tp,d in batches[lab].pulses['N']:
    if start<=tp<=end:ax.axvline(tp,color='.5',ls=':',lw=.8)
   ax.set(title=lab+(' · Calibración previa CO₂' if lab in LABS[:3] else ' · Holdout previo CO₂'),
    xlabel='Tiempo desde inoculación [h]',ylabel='CO₂ [g/(L·h)]',xlim=(start,end));ax.set_ylim(bottom=0);ax.grid(alpha=.25)
  hh,ll=axes.flat[0].get_legend_handles_labels()
  fig.suptitle('M original · '+label+' · '+window+'\nUn parámetro por escenario · transferencia y theta_nv2 fijos · O₂ activo',y=.99)
  fig.legend(hh,ll,loc='upper center',bbox_to_anchor=(.5,.89),ncol=3,title='Datos y escenarios · líneas verticales: nutrición')
  fig.subplots_adjust(top=.75,hspace=.4,wspace=.25);plt.show()


In [ ]:
assert theta==theta_before and calls==6
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==h for p,h in before.items())
print('Verificado: M original, un parámetro por escenario, transferencia fija, cero optimizaciones y ningún core0.')
print('Máximo error de balance',max(errors))
print('SUMMARY_JSON')
print(summary.reset_index().to_json(orient='records',force_ascii=False))


## Lectura de las pruebas

La sensibilidad sólo de A ya existía en10, pero con otra transferencia y sin O₂. La variación individual de tiempos faltaba; el ajuste conjunto09 no equivalía a esta prueba. Aquí toda la comparación usa una única referencia con O₂ activo, k=0.587042h⁻¹ y sin recalibraciones.

RMSE ordinario medio de LAB016–018, g/(L·h); cada fila cambia sólo el parámetro indicado:

| Escenario | Registro | Pico posterior | Cola tardía |
|---|---:|---:|---:|
| Referencia: A=.3, tau_r=4, tau_d=12 | 0.137399 | 0.227531 | 0.069878 |
| A=0 | 0.108247 | 0.101755 | 0.073418 |
| A=0.1 | 0.112127 | 0.126129 | 0.072181 |
| A=0.2 | 0.122439 | 0.172658 | 0.071001 |
| tau_rise_h=1 | 0.124839 | 0.182527 | 0.071113 |
| tau_rise_h=2 | 0.130031 | 0.201966 | 0.070670 |
| tau_rise_h=8 | 0.145015 | 0.251888 | 0.068527 |
| tau_decay_h=4 | 0.112187 | 0.126310 | 0.073371 |
| tau_decay_h=8 | 0.127288 | 0.191606 | 0.072288 |
| tau_decay_h=24 | 0.150777 | 0.269447 | 0.062539 |

Reducir A disminuye el refuerzo sin cambiar la forma del kernel normalizado. Variar tau_r o tau_d cambia el momento y la duración del refuerzo, además de su área integrada, aunque su máximo normalizado permanezca igual. Los tiempos del máximo del kernel no son automáticamente los del máximo de caudal: también intervienen química, actividad y depósito.

Acortar tau_r o tau_d mejora el exceso de pico del holdout frente a la referencia A=0.3/4h/12h. No elimina todos los errores temporales ni implica una mejora uniforme por LAB. Alargar tau_d a24h sostiene algo más la cola de LAB016–018, pero aumenta el pico y el error del registro completo. Sigue existiendo el compromiso entre pico y cola; LAB021 conserva un comportamiento distinto, con pico insuficiente y cola excesiva. Se incluyen métricas y figuras por LAB para no ocultarlo con promedios.

A=0 anula M y reproduce exactamente el caso sin transitorio del11 con O₂. Las pruebas de tiempos usan A=0.3, pues variar tiempos con A=0 sería inoperante. No se eligió un valor óptimo a partir de este holdout ya explorado; no se afirma mejora global ni identificación física de estos parámetros. Los originales permanecen intactos y todos los resultados nuevos quedan sólo en este SOURCE/EXECUTED.
